In [10]:
import os
import ast
import numpy as np
import pandas as pd
import psycopg2
from dotenv import load_dotenv
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import joblib

load_dotenv()

DB_CONFIG = {
    "dbname": os.getenv("DB_NAME"),
    "user": os.getenv("DB_USER"),
    "password": os.getenv("DB_PASSWORD"),
    "host": os.getenv("DB_HOST"),
    "port": os.getenv("DB_PORT")
}

SYMBOL = "2222"
OUTPUT_DIR = SYMBOL
os.makedirs(OUTPUT_DIR, exist_ok=True)

conn = psycopg2.connect(**DB_CONFIG)
conn.autocommit = True

print(f"Building pipeline for: {SYMBOL}")

Building pipeline for: 2222


In [ ]:
# LOAD PRICES
prices_query = f"""
SELECT date, open, high, low, close,
       volume, turnover, num_trades,
       change, change_pct
FROM public.prices
WHERE symbol = '{SYMBOL}'
ORDER BY date ASC;
"""

prices_df = pd.read_sql_query(prices_query, conn)
prices_df['date'] = pd.to_datetime(prices_df['date'])

# CREATE LAGS
prices_df['price_1y'] = prices_df['close'].shift(252)
prices_df['price_3m'] = prices_df['close'].shift(63)
prices_df['price_1m'] = prices_df['close'].shift(21)
prices_df['price_5d'] = prices_df['close'].shift(5)


prices_df['target_price_today'] = prices_df['close']

/tmp/ipykernel_114560/794225821.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  prices_df = pd.read_sql_query(prices_query, conn)


In [12]:
# LOAD EMBEDDINGS
emb_query = """
SELECT news_date::date AS date, embedding
FROM embeddings
ORDER BY news_date ASC;
"""

emb_df = pd.read_sql_query(emb_query, conn)
emb_df['date'] = pd.to_datetime(emb_df['date'])

/tmp/ipykernel_114560/2929506403.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  emb_df = pd.read_sql_query(emb_query, conn)


In [13]:
# MERGE
merged = pd.merge(prices_df, emb_df, on='date', how='inner')
merged = merged.dropna()

print("Merged shape:", merged.shape)

Merged shape: (11443, 20)


In [14]:
# ✅ FIX EMBEDDINGS (THIS WAS YOUR ERROR)

# convert string to list
merged['embedding'] = merged['embedding'].apply(ast.literal_eval)

# expand into columns
embedding_matrix = pd.DataFrame(
    merged['embedding'].tolist(),
    index=merged.index
)

embedding_matrix.columns = [f'emb_{i}' for i in range(embedding_matrix.shape[1])]

# drop original column
merged = merged.drop(columns=['embedding'])

# combine
merged = pd.concat([merged, embedding_matrix], axis=1)

print("After embedding expansion:", merged.shape)

After embedding expansion: (11443, 1555)


In [15]:
# TRAIN MODEL
feature_cols = [
    'price_1y',
    'price_3m',
    'price_1m',
    'price_5d',
    'price_4d',
    'price_3d',
    'price_2d',
    'price_1d'
]

embedding_cols = [col for col in merged.columns if col.startswith("emb_")]

X = merged[feature_cols + embedding_cols]
y = merged['target_price_today']

split_index = int(len(merged) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]
y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = Ridge(alpha=10)
model.fit(X_train_scaled, y_train)

train_r2 = r2_score(y_train, model.predict(X_train_scaled))
test_r2 = r2_score(y_test, model.predict(X_test_scaled))

print("Train R²:", round(train_r2,4))
print("Test R² :", round(test_r2,4))

Train R²: 0.9764
Test R² : 0.6467


In [16]:
baseline_pred = X_test['price_1d'].values
baseline_r2 = r2_score(y_test, baseline_pred)
print("Baseline R² (just yesterday price):", baseline_r2)

Baseline R² (just yesterday price): 0.7288679218128914


this means : Stock prices are highly autocorrelated .. price_today ≈ price_yesterday

ok , Does the model improve directional prediction?



In [17]:
y_pred = model.predict(X_test_scaled)

direction_accuracy = np.mean(
    np.sign(y_pred - X_test['price_1d'].values)
    ==
    np.sign(y_test.values - X_test['price_1d'].values)
)

print("Direction Accuracy:", direction_accuracy)

Direction Accuracy: 0.44997815640017474


this means .. my model is worse than just predicting yesterday’s price
Worse than random direction (50%)

New Discision : Instead of predicting close_t , let' predict (close_t / close_{t-1} - 1) to remove the autocorrelation advantage.



In [18]:
# ==========================================
# PREDICT RETURN INSTEAD OF PRICE
# ==========================================

# Create return target
merged['target_return'] = (
    merged['target_price_today'] / merged['price_1d'] - 1
)

# Drop any NaNs created
merged = merged.dropna()
print("Merged shape:", merged.shape)

# Features remain the same
X = merged[feature_cols + embedding_cols]
y = merged['target_return']

# Time split (same logic)
split_index = int(len(merged) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train model
model = Ridge(alpha=10)
model.fit(X_train_scaled, y_train)

# Predictions
y_pred_train = model.predict(X_train_scaled)
y_pred_test = model.predict(X_test_scaled)

# R²
train_r2 = r2_score(y_train, y_pred_train)
test_r2 = r2_score(y_test, y_pred_test)

print("===== RETURN MODEL RESULTS =====")
print("Train R²:", round(train_r2, 4))
print("Test R² :", round(test_r2, 4))


# ==========================================
# DIRECTIONAL ACCURACY
# ==========================================

direction_accuracy = np.mean(
    np.sign(y_pred_test) == np.sign(y_test)
)

print("Direction Accuracy:", round(direction_accuracy, 4))


# ==========================================
# SIMPLE LONG-ONLY STRATEGY
# Go long if predicted return > 0
# ==========================================

strategy_returns = np.where(y_pred_test > 0, y_test, 0)

mean_return = np.mean(strategy_returns)
std_return = np.std(strategy_returns)

sharpe = mean_return / std_return * np.sqrt(252) if std_return != 0 else 0

print("Average Strategy Daily Return:", round(mean_return, 6))
print("Strategy Sharpe Ratio:", round(sharpe, 4))


# ==========================================
# BASELINE: Always Hold Stock
# ==========================================

buy_hold_returns = y_test

bh_mean = np.mean(buy_hold_returns)
bh_std = np.std(buy_hold_returns)
bh_sharpe = bh_mean / bh_std * np.sqrt(252)

print("\n===== BASELINE (BUY & HOLD) =====")
print("Buy & Hold Sharpe:", round(bh_sharpe, 4))

Merged shape: (11443, 1556)
===== RETURN MODEL RESULTS =====
Train R²: 0.2107
Test R² : -0.3648
Direction Accuracy: 0.453
Average Strategy Daily Return: 0.000686
Strategy Sharpe Ratio: 1.3833

===== BASELINE (BUY & HOLD) =====
Buy & Hold Sharpe: 3.4526


 R² of −0.36 is worse than a naive mean-prediction baseline, and your direction accuracy of 45% is below a random coin flip. These are not just "weak results" — they are informative signals that something is structurally wrong, not just that the news signal is weak. The Sharpe of 1.38 vs. buy-and-hold's 3.45 confirms the model adds no value over passively holding TASI. Before trying fancier models, you need to fix the foundations.

In [21]:
# ==========================================
# PREDICT RETURN USING YESTERDAY'S NEWS
# ==========================================

# Create return target (Return_t)
merged['target_return'] = (
    merged['target_price_today'] / merged['price_1d'] - 1
)

# Shift embeddings by 1 day (News_{t-1})
for col in embedding_cols:
    merged[col] = merged[col].shift(1)

# Drop NaNs created by shifting
merged = merged.dropna()

# Features = yesterday's news only
X = merged[embedding_cols]
y = merged['target_return']

# Time-based split
split_index = int(len(merged) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train model
model = Ridge(alpha=10)
model.fit(X_train_scaled, y_train)

# Predictions
y_pred_train = model.predict(X_train_scaled)
y_pred_test = model.predict(X_test_scaled)

# R²
train_r2 = r2_score(y_train, y_pred_train)
test_r2 = r2_score(y_test, y_pred_test)

print("===== RETURN MODEL (News_{t-1} → Return_t) =====")
print("Train R²:", round(train_r2, 4))
print("Test R² :", round(test_r2, 4))


# ==========================================
# DIRECTIONAL ACCURACY
# ==========================================

direction_accuracy = np.mean(
    np.sign(y_pred_test) == np.sign(y_test)
)

print("Direction Accuracy:", round(direction_accuracy, 4))


# ==========================================
# SIMPLE LONG-ONLY STRATEGY
# ==========================================

strategy_returns = np.where(y_pred_test > 0, y_test, 0)

mean_return = np.mean(strategy_returns)
std_return = np.std(strategy_returns)

sharpe = mean_return / std_return * np.sqrt(252) if std_return != 0 else 0

print("Average Strategy Daily Return:", round(mean_return, 6))
print("Strategy Sharpe Ratio:", round(sharpe, 4))


# ==========================================
# BASELINE: Always Hold
# ==========================================

buy_hold_returns = y_test

bh_mean = np.mean(buy_hold_returns)
bh_std = np.std(buy_hold_returns)
bh_sharpe = bh_mean / bh_std * np.sqrt(252)

print("\n===== BASELINE (BUY & HOLD) =====")
print("Buy & Hold Sharpe:", round(bh_sharpe, 4))

===== RETURN MODEL (News_{t-1} → Return_t) =====
Train R²: 0.1817
Test R² : -0.2096
Direction Accuracy: 0.5046
Average Strategy Daily Return: 0.001467
Strategy Sharpe Ratio: 2.3389

===== BASELINE (BUY & HOLD) =====
Buy & Hold Sharpe: 3.4526


In [20]:
# SAVE MODEL

joblib.dump(model, f"{OUTPUT_DIR}/ridge_model.pkl")
joblib.dump(scaler, f"{OUTPUT_DIR}/scaler.pkl")

print("✅ Model saved.")

conn.close()

✅ Model saved.
